# Lab 14: Controlled Generation and Advanced Few-Shot (Gemini 3.1)

Even without fine-tuning, you can achieve extremely precise behavior by combining **System Instructions**, **Strict Schema Enforcement**, and **Few-Shot Prompting**. In this lab, we will master the art of controlling Gemini's output.

## Objectives
1. Implement **Few-Shot Learning** using structured examples.
2. Use **Strict JSON Enforcement** to eliminate model "chatter".
3. Apply **Negative Constraints** in System Instructions.
4. Enable **Thinking Mode** to see the model's internal reasoning.
5. Use **Response Hinting** to prefill the model's output.

In [ ]:
import os

from dotenv import load_dotenv
from google import genai
from google.genai import types
from pydantic import BaseModel

load_dotenv()
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

print("Client initialized for Controlled Generation.")

## 1. Advanced Few-Shot Prompting

Few-shot prompting is the most powerful way to teach Gemini a new pattern without fine-tuning. We provide pairs of examples directly in the `contents` list.

In [ ]:
examples = [
    types.Content(role="user", parts=[types.Part.from_text(
        text="Input: The battery life is great but the screen is dim."
    )]),
    types.Content(role="model", parts=[types.Part.from_text(
        text="POS: Battery Life | NEG: Screen Brightness"
    )]),
    types.Content(role="user", parts=[types.Part.from_text(
        text="Input: Fast shipping, but the box arrived crushed."
    )]),
    types.Content(role="model", parts=[types.Part.from_text(
        text="POS: Shipping Speed | NEG: Packaging Quality")]),
]

new_query = types.Content(role="user", parts=[types.Part.from_text(
    text="Input: The customer service was helpful, but I waited 2 hours on hold."
)])

response = client.models.generate_content(
    model="gemini-3.1-flash-lite-preview",
    contents=examples + [new_query]
)

print(f"Few-shot response: {response.text}")

## 2. Strict JSON Enforcement (No Chatter)

Gemini 3.1 is highly optimized for JSON. By using `response_mime_type`, you ensure that the model returns **only** valid JSON, with no introductory text like "Sure, here is your JSON...".

In [ ]:
class MovieInfo(BaseModel):
    title: str
    year: int
    director: str

prompt = "Tell me about the movie Inception."

response = client.models.generate_content(
    model="gemini-3.1-flash-lite-preview",
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=MovieInfo,
        temperature=0.1, # Keep it deterministic
    ),
    contents=prompt
)

print("Raw Response (Guaranteed pure JSON):")
print(response.text)

## 3. Negative Constraints and Logic

Sometimes, what the model *shouldn't* do is as important as what it *should* do. We use System Instructions to define these boundaries.

In [ ]:
system_instruction = """
You are a technical support bot.
CONSTRAINTS:
1. NEVER use technical jargon like 'kernel', 'BIOS', or 'latency'.
2. If the user asks about price, say 'I cannot discuss pricing'.
3. Keep answers under 20 words.
"""

def test_constraints(user_input):
    response = client.models.generate_content(
        model="gemini-3.1-flash-lite-preview",
        config=types.GenerateContentConfig(system_instruction=system_instruction),
        contents=user_input
    )
    print(f"User: {user_input}")
    print(f"Bot: {response.text}\n")

test_constraints("Why is my internet so slow?")
test_constraints("How much does the Pro plan cost?")

## 4. Thinking Mode (Internal Reasoning)

The 3.1 and 2.0 series support **Thinking Mode**, which exposes the model's "thoughts" before it provides the final answer. This is extremely helpful for complex logical tasks.

In [ ]:
config = types.GenerateContentConfig(
    thinking_config=types.ThinkingConfig(include_thoughts=True)
)

prompt = "Do you think ants have a consciousness? Explain your reasoning."

response = client.models.generate_content(
    model="gemini-3.1-flash-lite-preview",
    config=config,
    contents=prompt
)

for part in response.candidates[0].content.parts:
    if part.thought:
        print(f"--- MODEL THOUGHTS ---\n{part.text}\n")
    if part.text:
        print(f"--- FINAL RESPONSE ---\n{part.text}")

## 5. Response Hinting (Prefilling)

Response hinting involves starting the model's response for it by adding a `model` message at the end of the history. This forces the model to follow a specific tone or format right from the first token.

In [ ]:
contents = [
    types.Content(role="user", parts=[types.Part.from_text(
        text="Translate 'Hello' to French."
    )]),
    types.Content(role="model", parts=[types.Part.from_text(
        text="The French translation is: "
    )])
]

response = client.models.generate_content(
    model="gemini-3.1-flash-lite-preview",
    contents=contents
)

print(f"Prefilled Response: The French translation is: {response.text}")

## Summary

Control is the key to production-ready AI:
1. **Few-Shot**: Teaches by example without modifying model weights.
2. **Strict JSON**: Ensures seamless integration with software backends.
3. **Instructions**: Defines logical and negative boundaries.
4. **Thinking Mode**: Exposes internal logic for complex reasoning.
5. **Hinting**: Prefills output to force formatting and tone.